## 1. Get Dataset

In [1]:
import os
import sys
from pathlib import Path
import scanpy as sc

project_root = Path().absolute().parent
# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

import pertdata as pt  # noqa: E402

# Use the existing norman dataset location
norman = pt.PertDataset(name="norman", cache_dir_path="../data", silent=False)

print(norman)

Dataset already cached: d:\IML\Genomic-Data-Science\data\norman
Loading: d:\IML\Genomic-Data-Science\data\norman\norman\perturb_processed.h5ad
PertDataset object
    name: norman
    cache_dir_path: d:\IML\Genomic-Data-Science\data
    path: d:\IML\Genomic-Data-Science\data\norman
    adata: AnnData object with n_obs ✕ n_vars = 91205 ✕ 5045


## 2. Load the data


In [2]:
adata = sc.read_h5ad('../data/norman/norman/perturb_processed.h5ad')
print(f"Dataset loaded: {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"Available metadata: {list(adata.obs.columns)}")

Dataset loaded: 91205 cells × 5045 genes
Available metadata: ['condition', 'cell_type', 'dose_val', 'control', 'condition_name']


## 3. Prepare Data

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Keep data SPARSE - do not convert to dense
X = adata.X  # Keep as sparse matrix
print(f"Gene expression matrix: {X.shape} (sparse: {type(X).__name__})")

y_labels = adata.obs['condition'].values
print(f"Labels shape: {y_labels.shape}")
print(f"Unique perturbations: {len(set(y_labels))}")
print(f"Sample labels: {y_labels[:5]}")

# Encode string labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)
print(f"\nEncoded labels range: {y.min()} to {y.max()}")

# Split into train/test (80/20) - NO SUBSAMPLING, using full dataset
print("\n" + "="*50)
print("Using FULL dataset (sparse format)")
print("="*50)
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_seed, stratify=y
)

print(f"\nTraining data: X_train.shape={X_train.shape}")
print(f"Test data: X_test.shape={X_test.shape}")
print(f"Memory usage: ~{X_train.data.nbytes / 1e6:.1f} MB (sparse)")


Gene expression matrix: (91205, 5045) (sparse: csr_matrix)
Labels shape: (91205,)
Unique perturbations: 284
Sample labels: ['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl']
Categories (284, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

Encoded labels range: 0 to 283

Using FULL dataset (sparse format)

Training data: X_train.shape=(72964, 5045)
Test data: X_test.shape=(18241, 5045)
Memory usage: ~119.4 MB (sparse)


## 4. Classification

In [ ]:
from src.models.svm_model import LinearSVMClassifier
import time
''' C: 0.008416315395715555
  loss: squared_hinge
  max_iter: 909
  tol: 0.0007056401403169878'''
model = LinearSVMClassifier(
    C = 0.008416315395715555,
    max_iter = 909,
    tol = 0.0007056401403169878,
    random_state = 42,
    verbose = 1,                   
)
print("\n" + "H" * 50)
print("TRAINING SVM")
print("="*50)
start_time = time.time()
model.fit(X_train, y_train)
training_time = time.time() - start_time
results = model.evaluate(X_test, y_test)
print("\n" + "H" * 50)
print("R E S U L T")
print("="*50)
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")



HHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHH
TRAINING SVM
[LibLinear]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

print("\n" + "="*70)
print("ROC CURVE ANALYSIS (Multi-class)")
print("="*70)

# Get decision function scores for all classes
y_scores = model.predict_decision_function(X_test)

# Binarize the labels (one-vs-rest)
n_classes = len(np.unique(y_train))
y_test_bin = label_binarize(y_test, classes=range(n_classes))

print(f"\nNumber of classes: {n_classes}")
print(f"Test samples: {len(y_test)}")

# Compute ROC curve and ROC area for each class
fpr = dict() # false positive rate
tpr = dict() # true positive rate
roc_auc = dict() # area under the curve

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_scores[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_scores.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Compute macro-average ROC curve
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

print(f"\nROC AUC Scores:")
print(f"  Micro-average: {roc_auc['micro']:.4f}")
print(f"  Macro-average: {roc_auc['macro']:.4f}")

# Plot ROC curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Micro and Macro average
ax1 = axes[0]
ax1.plot(fpr["micro"], tpr["micro"],
         label=f'Micro-average ROC (AUC = {roc_auc["micro"]:.3f})',
         color='deeppink', linestyle=':', linewidth=3)
ax1.plot(fpr["macro"], tpr["macro"],
         label=f'Macro-average ROC (AUC = {roc_auc["macro"]:.3f})',
         color='navy', linestyle=':', linewidth=3)
ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve: Micro & Macro Average', fontsize=14, fontweight='bold')
ax1.legend(loc="lower right", fontsize=10)
ax1.grid(alpha=0.3)

# Right plot: Per-class ROC curves (top 10 most frequent classes)
ax2 = axes[1]

# Get top 10 most frequent classes
class_counts = np.bincount(y_train)
top_classes = np.argsort(class_counts)[-10:][::-1]

# Use a colormap for different classes
colors = plt.cm.tab10(np.linspace(0, 1, len(top_classes)))

for idx, class_id in enumerate(top_classes):
    class_label = label_encoder.inverse_transform([class_id])[0]
    ax2.plot(fpr[class_id], tpr[class_id], color=colors[idx],
            linewidth=2, alpha=0.7,
            label=f'{class_label[:20]}... (AUC={roc_auc[class_id]:.2f})')

ax2.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.3)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title(f'ROC Curve: Top 10 Classes', fontsize=14, fontweight='bold')
ax2.legend(loc="lower right", fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/roc_curve_multiclass.png', dpi=150, bbox_inches='tight')
print(f"\n✓ ROC curves saved to ../docs/roc_curve_multiclass.png")
plt.show()

# Show per-class AUC for all classes
print(f"\n" + "="*70)
print(f"Per-Class ROC AUC Scores (Top 20):")
print("="*70)
class_auc_sorted = sorted([(i, roc_auc[i]) for i in range(n_classes)], 
                           key=lambda x: x[1], reverse=True)
for rank, (class_id, auc_score) in enumerate(class_auc_sorted[:20], 1):
    class_label = label_encoder.inverse_transform([class_id])[0]
    samples = (y_test == class_id).sum()
    print(f"  {rank:2d}. {class_label:30s}: AUC={auc_score:.4f} ({samples:3d} test samples)")

## 4. Hierarchical Classification Setup

Create datasets for a 2-stage classification approach:
1. **Stage 1**: Binary classifier - "Most Common" vs "Rest"
2. **Stage 2**: Multi-class classifier - Classify within "Rest" samples only

In [ ]:
import pandas as pd

print("="*70)
print("HIERARCHICAL CLASSIFICATION: 2-STAGE APPROACH")
print("="*70)

# Analyze label distribution to find most common perturbation
label_counts = pd.Series(y_labels).value_counts()
most_common_label = label_counts.index[0]
most_common_count = label_counts.values[0]

print(f"\n Label Distribution Analysis:")
print(f"  Total perturbations: {len(label_counts)}")
print(f"  Most common: '{most_common_label}' ({most_common_count} samples, {most_common_count/len(y_labels)*100:.1f}%)")
print(f"  All others: {len(y_labels) - most_common_count} samples ({(len(y_labels) - most_common_count)/len(y_labels)*100:.1f}%)")

print(f"\n  Top 10 perturbations:")
for i, (label, count) in enumerate(label_counts.head(10).items(), 1):
    pct = (count / len(y_labels)) * 100
    marker = " ← MOST COMMON" if label == most_common_label else ""
    print(f"    {i:2d}. {label:30s}: {count:5d} samples ({pct:5.2f}%){marker}")

# Get encoded value for most common label
most_common_encoded = label_encoder.transform([most_common_label])[0]
print(f"\n  Most common encoded value: {most_common_encoded}")

### 4.1 Stage 1: Binary Classification ("Most Common" vs "Rest")

In [ ]:
print("\n" + "="*70)
print("STAGE 1 DATA: Binary Classification")
print("="*70)

# Create binary labels: 1 = most common, 0 = rest
y_binary = (y == most_common_encoded).astype(int)
y_labels_binary = np.where(y == most_common_encoded, most_common_label, "REST")

print(f"\n Binary Label Distribution:")
unique_binary, counts_binary = np.unique(y_binary, return_counts=True)
for val, count in zip(unique_binary, counts_binary):
    label = most_common_label if val == 1 else "REST"
    pct = (count / len(y_binary)) * 100
    print(f"  {val} ({label:30s}): {count:5d} samples ({pct:5.2f}%)")

# Split into train/test for Stage 1 binary classifier
X_train_stage1, X_test_stage1, y_train_stage1, y_test_stage1 = train_test_split(
    X, y_binary, test_size=0.2, random_state=random_seed, stratify=y_binary
)

print(f"\n✓ Stage 1 Train/Test Split (Binary):")
print(f"  Training: {X_train_stage1.shape[0]:5d} samples")
print(f"    - '{most_common_label}': {(y_train_stage1 == 1).sum():5d}")
print(f"    - 'REST': {(y_train_stage1 == 0).sum():5d}")
print(f"  Test:     {X_test_stage1.shape[0]:5d} samples")
print(f"    - '{most_common_label}': {(y_test_stage1 == 1).sum():5d}")
print(f"    - 'REST': {(y_test_stage1 == 0).sum():5d}")

### 4.2 Stage 2: Multi-class Classification (Within "Rest" Only)

In [ ]:
print("\n" + "="*70)
print("STAGE 2 DATA: Multi-class Classification (REST only)")
print("="*70)

# Filter to only "rest" samples (exclude most common)
rest_mask = y != most_common_encoded
X_rest = X[rest_mask]
y_rest_original = y[rest_mask]
y_labels_rest = y_labels[rest_mask]

print(f"\n📊 REST Subset Statistics:")
print(f"  Total 'REST' samples: {X_rest.shape[0]:5d}")
print(f"  Number of classes in REST: {len(np.unique(y_rest_original))}")
print(f"  Original total classes: {len(np.unique(y))}")
print(f"  Excluded: '{most_common_label}'")

# Re-encode labels for REST samples only (0 to n_classes-1)
label_encoder_rest = LabelEncoder()
y_rest = label_encoder_rest.fit_transform(y_labels_rest)

print(f"\n  REST labels re-encoded: {y_rest.min()} to {y_rest.max()}")

# Show top 10 classes in REST
rest_counts = pd.Series(y_labels_rest).value_counts()
print(f"\n  Top 10 perturbations in REST:")
for i, (label, count) in enumerate(rest_counts.head(10).items(), 1):
    pct = (count / len(y_labels_rest)) * 100
    print(f"    {i:2d}. {label:30s}: {count:5d} samples ({pct:5.2f}%)")

# Split into train/test for Stage 2 multi-class classifier (REST only)
X_train_stage2, X_test_stage2, y_train_stage2, y_test_stage2 = train_test_split(
    X_rest, y_rest, test_size=0.2, random_state=random_seed, stratify=y_rest
)

print(f"\n✓ Stage 2 Train/Test Split (REST multi-class):")
print(f"  Training: {X_train_stage2.shape[0]:5d} samples, {len(np.unique(y_train_stage2))} classes")
print(f"  Test:     {X_test_stage2.shape[0]:5d} samples, {len(np.unique(y_test_stage2))} classes")
print(f"  Features: {X_train_stage2.shape[1]:5d} genes")
print(f"  Memory:   ~{X_train_stage2.data.nbytes / 1e6:.1f} MB (sparse)")

### 4.3 Summary of All Datasets

In [ ]:
print("\n" + "="*70)
print("SUMMARY: ALL AVAILABLE DATASETS")
print("="*70)

print("\n🎯 ORIGINAL FULL MULTI-CLASS:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape} (classes: {len(np.unique(y_train))})")
print(f"  y_test:  {y_test.shape} (classes: {len(np.unique(y_test))})")
print(f"  Purpose: Classify ALL {len(np.unique(y))} perturbations directly")

print("\n🎯 STAGE 1 - BINARY CLASSIFIER:")
print(f"  X_train_stage1: {X_train_stage1.shape}")
print(f"  X_test_stage1:  {X_test_stage1.shape}")
print(f"  y_train_stage1: {y_train_stage1.shape} (binary: 0=REST, 1='{most_common_label}')")
print(f"  y_test_stage1:  {y_test_stage1.shape}")
print(f"  Purpose: Decide if sample is '{most_common_label}' or something else")

print("\n🎯 STAGE 2 - REST MULTI-CLASS CLASSIFIER:")
print(f"  X_train_stage2: {X_train_stage2.shape}")
print(f"  X_test_stage2:  {X_test_stage2.shape}")
print(f"  y_train_stage2: {y_train_stage2.shape} (classes: {len(np.unique(y_train_stage2))})")
print(f"  y_test_stage2:  {y_test_stage2.shape} (classes: {len(np.unique(y_test_stage2))})")
print(f"  Purpose: Classify which of the {len(np.unique(y_rest))} REST perturbations")

print("\n" + "="*70)
print("HIERARCHICAL CLASSIFICATION WORKFLOW:")
print("="*70)
print("1. Train binary classifier on (X_train_stage1, y_train_stage1)")
print(f"   → Predicts: Is it '{most_common_label}'? (YES/NO)")
print("\n2. If NO (prediction = REST):")
print("   → Pass to second classifier trained on (X_train_stage2, y_train_stage2)")
print(f"   → Predicts: Which of the {len(np.unique(y_rest))} REST perturbations?")
print("\n3. If YES:")
print(f"   → Final prediction = '{most_common_label}'")
print("\n💡 This approach may improve accuracy on the most common class")
print("   while still being able to classify all other perturbations.")